# EdgeYOLO-FPGA 板上渐进式验证 Notebook

本 notebook 按照从小到大的顺序逐步验证 FPGA 片上各模块，最终验证完整推理流程。

## 测试层级

| Level | 内容 | 说明 |
|-------|------|------|
| **L0** | 环境 & XDMA 连通性 | smoke test：HBM 读写、INST_BRAM 读写 |
| **L1** | 单算子（最小规模） | dcim_tiny、qa、dqa、im2col 基础尺寸 |
| **L2** | 单算子（网络真实规模） | extreme 规模 + YOLOv5n 各层真实尺寸 |
| **L3** | 组合算子 | mp、us、add、conv_pipeline 全链路 |
| **L4** | mini 网络 | 2-3 层 conv + residual add |
| **L5** | 完整网络 | YOLOv5n W8A8 全网推理到 model.23 |

## 数据通路（默认 staging=hbm）
```
Host ──XDMA──> HBM (act + weight + wb)
Host ──XDMA──> INST_BRAM (inst + HBM CDMA patch)
                   │
                   ├─ CDMA: HBM → tile_ibuf / VPU_BUF / WB
                   ├─ DCIM / VPU 计算
                   └─ CDMA: tile_obuf / VPU_BUF → HBM
Host <──XDMA── HBM (结果与 golden 逐 word 比对)
```

**运行前提**：FPGA 已烧录 bitstream；`xdma_info.exe` 可见设备；conda 环境 `chip_test_env`

---
## L0  环境 & XDMA 连通性

验证：
- Python 路径正确
- `xdma_rw.exe` 可找到
- 设备可寻址（读 DECODER_STATUS 寄存器）
- HBM 和 INST_BRAM 可读写

In [ ]:
import sys, struct, time
from pathlib import Path

# ── 路径设置 ──────────────────────────────────────────────────────
UNIT_TB_DIR = Path(".").resolve()
if UNIT_TB_DIR.name != "unit-tb":
    UNIT_TB_DIR = Path(__file__).resolve().parent if "__file__" in dir() else UNIT_TB_DIR
sys.path.insert(0, str(UNIT_TB_DIR))
REPO_ROOT = UNIT_TB_DIR.parent.parent.parent

print(f"Repo  : {REPO_ROOT}")
print(f"UnitTB: {UNIT_TB_DIR}")

# ── 导入核心模块 ───────────────────────────────────────────────────
from xdma_win import (
    XDMAWin, ChipRunnerWin, XDMA_RW_EXE,
    HBM_BASE, INST_BASE, REGS_BASE,
    REG_DECODER_STATUS, REG_INST_COUNT, REG_DECODER_CTRL,
)
from gen_data import generate_case, list_cases

assert XDMA_RW_EXE.exists(), f"xdma_rw.exe not found: {XDMA_RW_EXE}"
print(f"xdma_rw.exe: {XDMA_RW_EXE} ✓")

# ── 全局共享对象（整个 notebook 复用，避免重复创建） ────────────────
xdma   = XDMAWin(verbose=False)
runner = ChipRunnerWin(xdma=xdma, verbose=False)

# ── 全局结果记录表 ─────────────────────────────────────────────────
# 格式: [(level, name, staging, status, passed, total, note), ...]
TEST_RESULTS = []

def record(level, name, staging, passed_words, total_words, note=""):
    status = "PASS" if passed_words == total_words else "FAIL"
    TEST_RESULTS.append((level, name, staging, status, passed_words, total_words, note))
    icon = "✓" if status == "PASS" else "✗"
    print(f"  {icon} [{level}] {name} [{staging}]  {status}  ({passed_words}/{total_words}){' -- '+note if note else ''}")
    return status == "PASS"

print("\n全局对象初始化完成 ✓")

In [ ]:
# L0-A: XDMA 设备连通性 ── 读 DECODER_STATUS 寄存器
# 若 FPGA bitstream 正常烧录，STATUS 应为 0x00000002（done）或 0x00000000（idle）
status = xdma.read_u32(REGS_BASE + REG_DECODER_STATUS)
print(f"DECODER_STATUS = 0x{status:08x}")
assert status in (0x0, 0x2, 0x00000002), f"异常状态 0x{status:08x}，请检查 bitstream"
print("XDMA 通信正常 ✓")

# L0-B: HBM smoke test ── 写入特征值再读回
PATTERN_16B = bytes(range(16))
xdma.write(HBM_BASE + 0x0, PATTERN_16B)
rb = xdma.read(HBM_BASE + 0x0, 16)
assert rb == PATTERN_16B, f"HBM 读写不一致: {rb.hex()}"
print("HBM 读写正常 ✓")

# L0-C: INST_BRAM smoke test ── 写入 4 个已知 32-bit words 再读回
INST_PATTERN = struct.pack("<4I", 0x00000000, 0xF0000000, 0x12345678, 0xABCDEF01)
xdma.write(INST_BASE, INST_PATTERN)
rb2 = xdma.read(INST_BASE, 16)
assert rb2 == INST_PATTERN, f"INST_BRAM 读写不一致: {rb2.hex()}"
print("INST_BRAM 读写正常 ✓")

print("\n[L0] 环境检查 全部通过 ✓")

---
## L1  单算子最小规模

覆盖所有算子类型，使用已验证通过的最小参数（与 module_tb 仿真完全一致）。

| 算子 | Variant | 规模 |
|------|---------|------|
| DCIM matmul | dcim_tiny_1x1 | M=4 K=32 N=16 |
| DCIM matmul | conv6_s2_c3_to16 | M=36 K=108 N=16 (YOLOv5n model.0) |
| DCIM matmul | conv3_s2_c32_to64 | M=16 K=288 N=64 (4 tiles) |
| QA | qa_c16_signed | FP32→INT8, c=16 |
| DQA | dqa_c16_small | INT32→FP32, c=16 |
| im2col | im2col_6x6_s2_c3 | 6×6 stride=2 cin=3 |

In [ ]:
L1_CASES = [
    # (module,        variant,              quant,  timeout_s, note)
    ("dcim_matmul", "dcim_tiny_1x1",       "int8",  60,  "最小DCIM: M=4 K=32 N=16"),
    ("dcim_matmul", "conv6_s2_c3_to16",    "int8",  90,  "1-tile: model.0 真实层"),
    ("dcim_matmul", "conv3_s2_c32_to64",   "int8",  90,  "4-tile: model.3 真实层"),
    ("qa",          "qa_c16_signed",        "int8",  60,  "FP32→INT8 量化"),
    ("dqa",         "dqa_c16_small",        "int8",  60,  "INT32→FP32 反量化"),
    ("im2col",      "im2col_6x6_s2_c3",    "int8", 180,  "im2col stride=2 cin=3"),
]

print("=" * 65)
print("L1: 单算子最小规模 (staging=hbm)")
print("=" * 65)

for module, variant, quant, tmo, note in L1_CASES:
    run_dir = generate_case(module, variant, quant=quant)
    results = runner.run_case(run_dir, staging="hbm", timeout_s=tmo)
    for r in results:
        record("L1", f"{module}/{variant}", "hbm",
               r["passed"], r["total_words"], note)
        # 打印首个 mismatch 方便定位问题
        if not r["pass"] and r["first_mismatch"]:
            m = r["first_mismatch"]
            print(f"    ↳ word {m['word']}: exp={m['expected'][:16]}.. got={m['got'][:16]}..")

---
## L2  单算子网络真实规模

使用 YOLOv5n 各层的真实输入尺寸进行测试，验证极限规模下的正确性。

### L2-A  DCIM extreme & 网络层
覆盖 extreme 大尺寸（K=576/1152、N=512）以及网络中 M 最大（model.0: 25600）和 acc_depth 最深（model.8: acc=18）的层。

### L2-B  im2col 更大尺寸
覆盖 3x3 stride=2 c=32 和 1x1 c=512，验证 URAM pipeline 修复对所有尺寸的鲁棒性。

### L2-C  QA / DQA 更大通道数
覆盖 c=64/128 以及 INT16 累加模式，验证 VPU ready delay 对各规模的稳定性。

In [ ]:
# L2-A: DCIM extreme 规模 + 网络真实层
L2A_CASES = [
    # (module,        variant,                        quant,  tmo, note)
    ("dcim_matmul", "conv1_c64_to32",                 "int8",  90, "N=32 1x1 conv"),
    ("dcim_matmul", "conv3_c128_to128",               "int8",  90, "N=128 8-tile"),
    ("dcim_matmul", "extreme_int8_1x1_c512_to512",    "int8", 120, "N=512 32-tile pass"),
    ("dcim_matmul", "extreme_int8_3x3_c128_to512",    "int8", 120, "K=1152 N=512 acc=18"),
    ("dcim_matmul", "extreme_int8_6x6_c3_to64",       "int8", 120, "M=36 K=108 N=64"),
    ("dcim_matmul", "dcim_model_0_conv",               "int8", 180, "model.0 M=25600 真实尺寸"),
    ("dcim_matmul", "dcim_model_3_conv",               "int8", 180, "model.3 K=288 N=64 acc=5"),
    ("dcim_matmul", "dcim_model_7_conv",               "int8", 180, "model.7 K=1152 N=256 acc=18"),
]

print("=" * 65)
print("L2-A: DCIM extreme & 网络层 (staging=hbm)")
print("=" * 65)

for module, variant, quant, tmo, note in L2A_CASES:
    run_dir = generate_case(module, variant, quant=quant)
    results = runner.run_case(run_dir, staging="hbm", timeout_s=tmo)
    for r in results:
        record("L2-A", f"{module}/{variant}", "hbm",
               r["passed"], r["total_words"], note)
        if not r["pass"] and r["first_mismatch"]:
            m = r["first_mismatch"]
            print(f"    ↳ word {m['word']}: exp={m['expected'][:16]}.. got={m['got'][:16]}..")

In [ ]:
# L2-B: im2col 更大尺寸 ── 验证 URAM pipeline 修复对所有 im2col 变体的鲁棒性
L2B_CASES = [
    # im2col_3x3_s2_c32: stride=2 cin=32 → 代表网络中间层 3x3 conv
    ("im2col", "im2col_3x3_s2_c32",  "int8", 180, "3x3 stride=2 cin=32"),
    # im2col_3x3_s1_c128: stride=1 cin=128 → 代表深层 3x3 conv（无 padding 下采样）
    ("im2col", "im2col_3x3_s1_c128", "int8", 180, "3x3 stride=1 cin=128"),
    # im2col_1x1_c512: 1x1 conv 等价 im2col，验证大 cin 的 URAM 访问
    ("im2col", "im2col_1x1_c512",    "int8", 120, "1x1 cin=512 (pure reshape)"),
]

# L2-C: QA/DQA 更大通道数 ── 验证 VPU ready delay 对所有规模的稳定性
L2C_CASES = [
    ("dqa",  "dqa_c32_mid",           "int8",  60, "DQA cin=32"),
    ("dqa",  "dqa_c64_mid",           "int8",  60, "DQA cin=64"),
    ("dqa",  "dqa_c128_sppf",         "int8",  60, "DQA cin=128 (SPPF)"),
    ("qa",   "qa_c64_clip",           "int8",  60, "QA cin=64 clip"),
    ("qa",   "qa_c128_dense",         "int8",  60, "QA cin=128"),
]

print("=" * 65)
print("L2-B: im2col 大尺寸 (staging=hbm)")
print("=" * 65)
for module, variant, quant, tmo, note in L2B_CASES:
    run_dir = generate_case(module, variant, quant=quant)
    results = runner.run_case(run_dir, staging="hbm", timeout_s=tmo)
    for r in results:
        record("L2-B", f"{module}/{variant}", "hbm",
               r["passed"], r["total_words"], note)
        if not r["pass"] and r["first_mismatch"]:
            m = r["first_mismatch"]
            print(f"    ↳ word {m['word']}: exp={m['expected'][:16]}.. got={m['got'][:16]}..")

print()
print("=" * 65)
print("L2-C: QA/DQA 大通道数 (staging=hbm)")
print("=" * 65)
for module, variant, quant, tmo, note in L2C_CASES:
    run_dir = generate_case(module, variant, quant=quant)
    results = runner.run_case(run_dir, staging="hbm", timeout_s=tmo)
    for r in results:
        record("L2-C", f"{module}/{variant}", "hbm",
               r["passed"], r["total_words"], note)
        if not r["pass"] and r["first_mismatch"]:
            m = r["first_mismatch"]
            print(f"    ↳ word {m['word']}: exp={m['expected'][:16]}.. got={m['got'][:16]}..")

---
## L3  组合算子

验证 MaxPool、Upsample、Add、conv_pipeline（im2col→CDMA→DCIM→DQA→QA 全链路）等非 DCIM 算子。

- **mp**：MaxPool 5×5 s1（SPPF 结构）及 ResNet stem 3×3 s2
- **us**：Upsample 2×（PAN neck 上采样）
- **add**：Residual Add（identity shortcut）
- **conv_pipeline**：单层完整推理链路，验证各 IP 间握手时序

In [ ]:
L3_CASES = [
    # MaxPool: 5x5 s1 pad=2 (SPPF)  和  3x3 s2 pad=1 (ResNet stem)
    ("mp",    "mp_sppf_128_10",         "int8",  60, "MaxPool 5x5 c=128 hw=10"),
    ("mp",    "mp_resnet_stem",          "int8",  60, "MaxPool 3x3 s2 (ResNet)"),
    # Upsample 2x (nearest neighbor)
    ("us",    "us_128_10_to20",          "int8",  60, "Upsample 2x c=128 10→20"),
    ("us",    "us_64_20_to40",           "int8",  60, "Upsample 2x c=64 20→40"),
    # Residual Add
    ("add",   "add_residual_16",         "int8",  60, "Add residual c=16"),
    ("add",   "add_residual_32",         "int8",  60, "Add residual c=32"),
    ("add",   "add_pan_64",              "int8",  60, "Add PAN path c=64"),
    # conv_pipeline: im2col→CDMA→DCIM→DQA→QA 全链路
    ("conv_pipeline", "pipe_conv1_c16_to16",         "int8", 120, "1x1 conv 全链路"),
    ("conv_pipeline", "pipe_conv3_s2_c32_to64",      "int8", 120, "3x3 stride=2 全链路"),
    ("conv_pipeline", "pipe_conv1_c512_to64_tilepass","int8", 180, "1x1 c512→64 多 tile pass"),
]

print("=" * 65)
print("L3: 组合算子 (staging=hbm)")
print("=" * 65)

for module, variant, quant, tmo, note in L3_CASES:
    run_dir = generate_case(module, variant, quant=quant)
    results = runner.run_case(run_dir, staging="hbm", timeout_s=tmo)
    for r in results:
        record("L3", f"{module}/{variant}", "hbm",
               r["passed"], r["total_words"], note)
        if not r["pass"] and r["first_mismatch"]:
            m = r["first_mismatch"]
            print(f"    ↳ word {m['word']}: exp={m['expected'][:16]}.. got={m['got'][:16]}..")

---
## L4  Mini 网络

验证多层网络的指令流水线（多条 DCIM → Add → VPU 串联），关注：
- 层间数据搬运（tile_obuf → VPU_BUF → tile_ibuf）
- WAIT_DCIM / WAIT_VPU 屏障正确性
- 跨层 HBM drain 聚合读回

| Variant | 结构 |
|---------|------|
| mini_2conv_c16 | Conv→BN→ReLU → Conv→BN→ReLU，c=16 |
| mini_3conv_residual_c32 | Conv→Conv→Conv + residual add，c=32 |

In [ ]:
L4_CASES = [
    ("mini_network", "mini_2conv_c16",         "int8", 180, "2-conv 串联 c=16"),
    ("mini_network", "mini_3conv_residual_c32", "int8", 240, "3-conv + residual c=32"),
]

print("=" * 65)
print("L4: Mini 网络 (staging=hbm)")
print("=" * 65)

for module, variant, quant, tmo, note in L4_CASES:
    run_dir = generate_case(module, variant, quant=quant)
    results = runner.run_case(run_dir, staging="hbm", timeout_s=tmo)
    for r in results:
        record("L4", f"{module}/{variant}", "hbm",
               r["passed"], r["total_words"], note)
        if not r["pass"] and r["first_mismatch"]:
            m = r["first_mismatch"]
            print(f"    ↳ word {m['word']}: exp={m['expected'][:16]}.. got={m['got'][:16]}..")

---
## L5  完整网络：YOLOv5n W8A8

调用编译器对完整 YOLOv5n（到 model.23）生成 ISA binary，在 FPGA 上执行，验证：
1. 编译器生成的完整指令流能在硬件上跑通
2. 最终输出 feature map 非全零、无 NaN/Inf
3. 输出统计量（range / mean / std）与 golden 量级一致

> **注意**：此步骤需要 `model/yolov5n/` 目录下有解析好的量化权重（`network.json` + `weights/*.npz`）。

### L5 运行参数

In [ ]:
import subprocess, json
import numpy as np

# ── L5 配置 ──────────────────────────────────────────────────────
BUILD_DIR   = REPO_ROOT / "tests" / "chip" / "dist" / "yolov5n_w8a8"
SKIP_COMPILE = BUILD_DIR.joinpath("plan.json").exists()   # 已编译则跳过
SEED        = 42        # 测试输入随机种子
TIMEOUT_S   = 300.0     # 完整网络推理超时（5 分钟）
# ─────────────────────────────────────────────────────────────────

BUILD_DIR.mkdir(parents=True, exist_ok=True)

# Step 1: 编译（如未编译）
if SKIP_COMPILE:
    print(f"[L5] plan.json 已存在，跳过编译。如需重新编译请删除 {BUILD_DIR}")
else:
    print("[L5] 开始编译 YOLOv5n W8A8...")
    cmd = [
        sys.executable,
        str(REPO_ROOT / "tests/chip/compiler/compile.py"),
        "--network", "yolov5n",
        "--out", str(BUILD_DIR),
        "--full", "--mode", "int8",
        "--hw-caps", str(REPO_ROOT / "tests/chip/compiler/lowering/hw_caps.yaml"),
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("编译失败！")
        print(r.stderr[-2000:])
        raise RuntimeError("Compile failed")
    print(r.stdout[-1000:])
    print("[L5] 编译完成 ✓")

# Step 2: 读取 plan
plan = json.loads((BUILD_DIR / "plan.json").read_text())
n_layers = plan["compile_meta"]["num_conv_layers_compiled"]
out_hw   = plan["host_io"]["output_hw"]
out_c    = plan["host_io"]["output_c"]
print(f"[L5] 编译层数: {n_layers}")
print(f"[L5] 输出 feature map: {out_hw[0]}x{out_hw[1]}x{out_c}")

In [ ]:
sys.path.insert(0, str(REPO_ROOT / "tests" / "chip"))
from runtime.xdma_driver import XDMA, ChipRunner

# Step 3: 生成测试输入（640×640×3 随机 UINT8）
rng = np.random.default_rng(SEED)
input_nhwc = rng.integers(0, 256, size=(640, 640, 3), dtype=np.uint8)
input_bin  = input_nhwc.tobytes()
print(f"[L5] 输入: shape={input_nhwc.shape}  bytes={len(input_bin):,}")

# Step 4: 上板执行
# ChipRunner 使用 xdma_win 的 H2C/C2H 通道；Windows 下设备路径为 xdma0_user
DEVICE = "xdma0_user"  # Windows XDMA 设备路径（由 xdma_driver 自动适配）

print(f"[L5] 开始执行，timeout={TIMEOUT_S}s ...")
t0 = time.time()
try:
    with XDMA(DEVICE) as dev:
        cr = ChipRunner(dev)
        result_bytes = cr.run_plan(BUILD_DIR, input_bin, timeout_s=TIMEOUT_S)
    elapsed = time.time() - t0

    # Step 5: 解析并校验输出
    result = np.frombuffer(result_bytes, dtype=np.float32).reshape(out_hw[0], out_hw[1], out_c)
    print(f"[L5] 执行完成  耗时={elapsed:.1f}s")
    print(f"[L5] 输出 shape: {result.shape}  dtype: {result.dtype}")
    print(f"[L5] range: [{result.min():.4f}, {result.max():.4f}]")
    print(f"[L5] mean:  {result.mean():.4f}   std: {result.std():.4f}")

    # 基本 sanity checks
    assert result.shape == (out_hw[0], out_hw[1], out_c), "shape 不匹配"
    assert not np.all(result == 0),       "输出全零！计算未执行"
    assert not np.any(np.isnan(result)),  "输出含 NaN"
    assert not np.any(np.isinf(result)),  "输出含 Inf"

    # 保存供后续分析
    np.save(BUILD_DIR / "output_features.npy", result)
    print(f"[L5] 结果已保存至 {BUILD_DIR}/output_features.npy")

    record("L5", "yolov5n_w8a8/full", "hbm", result.size, result.size,
           f"range=[{result.min():.2f},{result.max():.2f}] mean={result.mean():.3f}")
    print("\n[L5] 完整网络验证 PASS ✓")

except Exception as e:
    record("L5", "yolov5n_w8a8/full", "hbm", 0, 1, str(e)[:80])
    print(f"\n[L5] 完整网络验证 FAIL: {e}")

---
## 汇总报告

汇总所有层级的测试结果，输出通过率表格和失败明细。

In [ ]:
from collections import defaultdict

# ── 按 level 统计 ─────────────────────────────────────────────────
level_stats = defaultdict(lambda: {"pass": 0, "fail": 0})
failures = []

for level, name, staging, status, pw, tw, note in TEST_RESULTS:
    if status == "PASS":
        level_stats[level]["pass"] += 1
    else:
        level_stats[level]["fail"] += 1
        failures.append((level, name, staging, pw, tw, note))

# ── 打印汇总表 ────────────────────────────────────────────────────
WIDTH = 68
print("=" * WIDTH)
print(f"{'PROGRESSIVE TEST REPORT':^{WIDTH}}")
print("=" * WIDTH)
print(f"  {'Level':<8} {'Pass':>5} {'Fail':>5} {'Rate':>8}  {'Status'}")
print("-" * WIDTH)

all_pass = True
for level in ["L0", "L1", "L2-A", "L2-B", "L2-C", "L3", "L4", "L5"]:
    s = level_stats.get(level)
    if s is None:
        print(f"  {level:<8} {'—':>5} {'—':>5} {'—':>8}  (未运行)")
        continue
    total = s["pass"] + s["fail"]
    rate  = s["pass"] / total * 100 if total else 0
    icon  = "✓" if s["fail"] == 0 else "✗"
    print(f"  {level:<8} {s['pass']:>5} {s['fail']:>5} {rate:>7.1f}%  {icon}")
    if s["fail"]:
        all_pass = False

print("-" * WIDTH)
total_p = sum(s["pass"] for s in level_stats.values())
total_f = sum(s["fail"] for s in level_stats.values())
total   = total_p + total_f
print(f"  {'TOTAL':<8} {total_p:>5} {total_f:>5} {total_p/total*100 if total else 0:>7.1f}%  "
      f"{'ALL PASS ✓' if all_pass else 'SOME FAILED ✗'}")
print("=" * WIDTH)

# ── 失败明细 ──────────────────────────────────────────────────────
if failures:
    print(f"\n{'失败明细':}")
    print("-" * WIDTH)
    for level, name, staging, pw, tw, note in failures:
        print(f"  [{level}] {name} [{staging}]  {pw}/{tw}  {note}")
else:
    print("\n所有已运行的测试全部通过！")

---
## 附录：单步调试工具

当某个 case FAIL 时，可在下方 cell 中逐步诊断。

In [ ]:
from xdma_win import (
    REGS_BASE, INST_BASE, VPU_BUF_BASE, TILE_IBUF_BASE, TILE_OBUF_BASE,
    TILE_IBUF_SIZE, TILE_OBUF_SIZE, DCIM_NUM_TILES,
    REG_DECODER_STATUS, REG_INST_COUNT, REG_DECODER_CTRL,
)

# ── 寄存器快照 ────────────────────────────────────────────────────
print("=== VPU_AXI_Regs ===")
print(f"  STATUS (0x04)         = 0x{xdma.read_u32(REGS_BASE + 0x04):08x}  [bit0=vpu_ready]")
print(f"  DECODER_STATUS (0x40) = 0x{xdma.read_u32(REGS_BASE + REG_DECODER_STATUS):08x}  "
      f"[bit0=busy bit1=done bit31=err]")
print(f"  INST_COUNT (0x3C)     = {xdma.read_u32(REGS_BASE + REG_INST_COUNT)}")

In [ ]:
from xdma_win import hex_to_bin
from hbm_flow import patch_inst_for_hbm, build_hbm_output_drain, OP_CDMA_COPY

# ── 手动运行指定 case 并诊断 ──────────────────────────────────────
# 修改这两行来指定要调试的 case
DBG_MODULE  = "dcim_matmul"
DBG_VARIANT = "conv3_s2_c32_to64"
DBG_QUANT   = "int8"

dbg_dir = generate_case(DBG_MODULE, DBG_VARIANT, quant=DBG_QUANT)
print(f"Run dir: {dbg_dir}\n")

# 打印 drain CDMA 地址（确认 tile → HBM slot 映射）
drain_insts, _ = build_hbm_output_drain(dbg_dir)
print("Drain CDMA 指令:")
i = 0
while i < len(drain_insts):
    word = drain_insts[i]
    op = (word >> 28) & 0xF
    if op == OP_CDMA_COPY and i + 5 < len(drain_insts):
        sm, sl = drain_insts[i+1], drain_insts[i+2]
        dm, dl = drain_insts[i+3], drain_insts[i+4]
        nb     = drain_insts[i+5]
        src = (sm << 32) | sl
        dst = (dm << 32) | dl
        print(f"  src=0x{src:011x} -> dst=0x{dst:011x}  {nb} bytes")
        i += 7
    else:
        i += 1

# 执行并显示结果
runner.verbose = True
results = runner.run_case(dbg_dir, staging="hbm", timeout_s=120.0)
runner.verbose = False
for r in results:
    s = "PASS" if r["pass"] else f"FAIL ({r['passed']}/{r['total_words']})"
    print(f"\n{r['name']}: {s}")
    if not r["pass"]:
        for mm in r["mismatches"][:10]:
            print(f"  word {mm['word']:3d}: exp={mm['expected'][:16]}.. got={mm['got'][:16]}..")

In [ ]:
from xdma_win import HBM_BASE, HBM_OFF_OUTPUT

# ── 直接读取任意 FPGA 地址（16-byte word 格式）────────────────────
# 修改 READ_ADDR 和 READ_WORDS 来检查任意区域
READ_ADDR  = HBM_BASE + HBM_OFF_OUTPUT   # ← 修改
READ_WORDS = 8                            # ← 读多少个 128-bit word

raw = xdma.read(READ_ADDR, READ_WORDS * 16)
print(f"addr=0x{READ_ADDR:011x}  ({READ_WORDS} × 128-bit words):")
for i in range(READ_WORDS):
    w = raw[i*16:(i+1)*16]
    # 显示为 4 个 u32 小端
    u32s = struct.unpack("<4I", w[:16])
    print(f"  [{i:3d}] {w.hex()}  ({' '.join(f'{v:08x}' for v in u32s)})")